# 📖 Notebook 2: Database Replication & Failover

## Why This Matters

In 2017, GitLab accidentally deleted a production database. They had backups,
but the restore took **18 hours**. During that time, they lost 6 hours of data.

With **streaming replication**, a standby database receives every change in
real-time. If the primary crashes, you promote the standby — and you are back
online in seconds, not hours.

This is how banks, stock exchanges, and cloud providers keep databases running 24/7.

## Learning Objectives

- Understand how PostgreSQL streaming replication works
- Monitor replication lag and WAL (Write-Ahead Log) status
- Perform a manual failover (promote standby to primary)
- Understand the split-brain problem and how to prevent it

## 🛠️ Setup

Make sure all services are running:

```bash
cd enterprise-patterns/bcdr
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).

In [1]:
import psycopg2
import subprocess
import time
from tabulate import tabulate

DB_PRIMARY = {
    "host": "localhost", "port": 5432,
    "database": "bcdr_demo", "user": "demo", "password": "demo"
}
DB_STANDBY = {
    "host": "localhost", "port": 5433,
    "database": "bcdr_demo", "user": "demo", "password": "demo"
}

def get_primary_connection():
    return psycopg2.connect(**DB_PRIMARY)

def get_standby_connection():
    return psycopg2.connect(**DB_STANDBY)

def docker_exec(container, cmd):
    """Run a command inside a Docker container."""
    result = subprocess.run(
        ["docker", "exec", container] + cmd,
        capture_output=True, text=True, timeout=30
    )
    return result.stdout.strip(), result.stderr.strip()

# Test connections
for name, cfg in [('Primary', DB_PRIMARY), ('Standby', DB_STANDBY)]:
    try:
        conn = psycopg2.connect(**cfg)
        conn.close()
        print(f"✅ {name} connected (port {cfg['port']})")
    except Exception as e:
        print(f"❌ {name} failed: {e}")

✅ Primary connected (port 5432)
✅ Standby connected (port 5433)


## 📚 How Streaming Replication Works

PostgreSQL streaming replication uses the **Write-Ahead Log (WAL)**:

```
1. Client sends a write (INSERT/UPDATE/DELETE) to the primary
2. Primary writes the change to the WAL file (on disk)
3. Primary sends the WAL records to all connected standbys
4. Standby receives WAL records and replays them
5. Standby now has the same data as the primary
```

### WAL (Write-Ahead Log)

The WAL is a sequential log of every change made to the database.
Think of it like a bank transaction log — before money moves,
the transaction is written to the log first.

This is the foundation of both replication AND backup recovery.

### Asynchronous vs Synchronous Replication

| Mode | How It Works | Trade-off |
|------|-------------|-----------|
| **Async** (default) | Primary does NOT wait for standby to confirm | Faster writes, but standby may be slightly behind |
| **Sync** | Primary waits for standby to confirm EACH write | Slower writes, but zero data loss (RPO = 0) |

In [2]:
# =============================================================================
# Demo: Inspect WAL Status on Primary
# =============================================================================

conn = get_primary_connection()
cur = conn.cursor()

# Current WAL position on primary
cur.execute("SELECT pg_current_wal_lsn(), pg_walfile_name(pg_current_wal_lsn())")
lsn, wal_file = cur.fetchone()

print("=" * 65)
print("WAL STATUS ON PRIMARY")
print("=" * 65)
print(f"  Current WAL LSN:  {lsn}")
print(f"  Current WAL file: {wal_file}")
print()
print("LSN = Log Sequence Number — a pointer to a position in the WAL.")
print("Each write advances the LSN. The standby tries to keep up.")

# Check replication slots
cur.execute(
    "SELECT slot_name, active, restart_lsn, confirmed_flush_lsn "
    "FROM pg_replication_slots"
)
slots = cur.fetchall()
print()
print("Replication Slots:")
for s in slots:
    print(f"  Slot: {s[0]}, Active: {s[1]}, Restart LSN: {s[2]}")
print()
print("💡 Replication slots prevent the primary from discarding WAL files")
print("   that the standby has not yet received.")

conn.close()

WAL STATUS ON PRIMARY
  Current WAL LSN:  0/65CB7D8
  Current WAL file: 000000010000000000000006

LSN = Log Sequence Number — a pointer to a position in the WAL.
Each write advances the LSN. The standby tries to keep up.

Replication Slots:
  Slot: standby_slot, Active: True, Restart LSN: 0/65CB7D8

💡 Replication slots prevent the primary from discarding WAL files
   that the standby has not yet received.


In [3]:
# =============================================================================
# Demo: Monitor Replication Lag
# =============================================================================

conn = get_primary_connection()
cur = conn.cursor()

cur.execute(
    "SELECT pid, client_addr, state, "
    "pg_wal_lsn_diff(sent_lsn, write_lsn) as send_lag_bytes, "
    "pg_wal_lsn_diff(sent_lsn, replay_lsn) as replay_lag_bytes, "
    "sync_state "
    "FROM pg_stat_replication"
)
rows = cur.fetchall()
conn.close()

print("=" * 65)
print("REPLICATION LAG MONITOR")
print("=" * 65)

if rows:
    table = []
    for r in rows:
        table.append([r[0], r[1], r[2], f"{r[3]} bytes", f"{r[4]} bytes", r[5]])
    print(tabulate(table,
        headers=["PID", "Address", "State", "Send Lag", "Replay Lag", "Sync"],
        tablefmt="grid"))
    print()
    print("💡 Send Lag = bytes sent but not yet written by standby")
    print("   Replay Lag = bytes sent but not yet replayed by standby")
    print("   Both should be 0 or very small during normal operation.")
else:
    print("⚠️  No standbys connected")

REPLICATION LAG MONITOR
+-------+---------------+-----------+------------+--------------+--------+
|   PID | Address       | State     | Send Lag   | Replay Lag   | Sync   |
+=======+===============+===========+============+==============+========+
|   107 | 192.168.192.7 | streaming | 0 bytes    | 0 bytes      | async  |
+-------+---------------+-----------+------------+--------------+--------+

💡 Send Lag = bytes sent but not yet written by standby
   Replay Lag = bytes sent but not yet replayed by standby
   Both should be 0 or very small during normal operation.


## 📚 Verifying Replication Works

The best way to verify replication is simple:
1. Write data to the primary
2. Read it from the standby
3. Confirm they match

The standby is **read-only** — you cannot write to it. This is a safety feature.

In [4]:
# =============================================================================
# Demo: Write to Primary, Read from Standby
# =============================================================================

# Write to primary
primary_conn = get_primary_connection()
primary_conn.autocommit = True
primary_cur = primary_conn.cursor()

marker = f"REPL_TEST_{int(time.time())}"
primary_cur.execute(
    "INSERT INTO audit_log (table_name, record_id, action, changed_by) "
    "VALUES (%s, %s, %s, %s) RETURNING id",
    ('repl_test', 999, 'INSERT', marker)
)
new_id = primary_cur.fetchone()[0]
print(f"✅ Wrote record #{new_id} to PRIMARY with marker: {marker}")

# Wait a moment for replication
time.sleep(0.5)

# Read from standby
standby_conn = get_standby_connection()
standby_cur = standby_conn.cursor()
standby_cur.execute(
    "SELECT id, table_name, action, changed_by FROM audit_log WHERE id = %s",
    (new_id,)
)
row = standby_cur.fetchone()

if row:
    print(f"✅ Read record #{row[0]} from STANDBY: {row[3]}")
    print("\n🎉 Replication is working! Data written to primary appears on standby.")
else:
    print("⚠️  Record not yet visible on standby (replication may be delayed)")

# Verify standby is read-only
try:
    standby_cur.execute(
        "INSERT INTO audit_log (table_name, record_id, action) "
        "VALUES ('test', 0, 'TEST')"
    )
    print("❌ Standby accepted a write — this should not happen!")
except Exception as e:
    print(f"\n✅ Standby correctly rejected write: {type(e).__name__}")
    print("   Hot standbys are READ-ONLY. This prevents split-brain.")
    standby_conn.rollback()

# Cleanup
primary_cur.execute("DELETE FROM audit_log WHERE changed_by = %s", (marker,))
primary_conn.close()
standby_conn.close()

✅ Wrote record #2 to PRIMARY with marker: REPL_TEST_1776712820


✅ Read record #2 from STANDBY: REPL_TEST_1776712820

🎉 Replication is working! Data written to primary appears on standby.

✅ Standby correctly rejected write: ReadOnlySqlTransaction
   Hot standbys are READ-ONLY. This prevents split-brain.


## 📚 The Split-Brain Problem

**Split-brain** is the most dangerous scenario in database failover:

```
  [Primary A] ←── network partition ──→ [Primary B]
       ↓                                      ↓
  accepts writes                        accepts writes
  (different data!)                     (different data!)
```

Both servers think they are the primary and accept writes independently.
The data **diverges** — and merging it back together is extremely difficult.

### Prevention Strategies

| Strategy | How It Works |
|----------|-------------|
| **Fencing (STONITH)** | Physically shut down the old primary before promoting standby |
| **Quorum** | Require majority agreement (odd number of nodes: 3, 5, 7) |
| **Watchdog Timer** | Node auto-shuts-down if it loses contact with the cluster |
| **Lease-based** | Primary must periodically renew a lease; if it cannot, standby takes over |

In this lab, we will do **manual failover with explicit fencing** (stopping the primary first).

## 📚 Manual Failover Procedure

Here is the step-by-step procedure to failover from primary to standby:

```
Step 1: FENCE the primary (stop it from accepting writes)
Step 2: VERIFY the standby has received all WAL records
Step 3: PROMOTE the standby to become the new primary
Step 4: UPDATE application connection strings
Step 5: VERIFY the new primary is accepting writes
```

**Important**: In a real production system, you would use tools like
**Patroni**, **pg_auto_failover**, or **Pacemaker** to automate this.
We do it manually here so you understand what happens under the hood.

In [5]:
# =============================================================================
# Demo: Check Standby Status Before Failover
# =============================================================================
# Before promoting, always check the standby is healthy and caught up.

standby_conn = get_standby_connection()
standby_cur = standby_conn.cursor()

# Is this server in recovery mode? (True = standby)
standby_cur.execute("SELECT pg_is_in_recovery()")
is_standby = standby_cur.fetchone()[0]

# Last WAL position received and replayed
standby_cur.execute(
    "SELECT pg_last_wal_receive_lsn(), "
    "pg_last_wal_replay_lsn(), "
    "pg_last_xact_replay_timestamp()"
)
received_lsn, replayed_lsn, last_replay_time = standby_cur.fetchone()
standby_conn.close()

print("=" * 65)
print("STANDBY STATUS CHECK")
print("=" * 65)
print(f"  Is in recovery (standby) mode: {is_standby}")
print(f"  Last WAL received:  {received_lsn}")
print(f"  Last WAL replayed:  {replayed_lsn}")
print(f"  Last replay time:   {last_replay_time}")

if is_standby:
    print("\n✅ Server is in standby mode — ready for promotion")
    if received_lsn == replayed_lsn:
        print("✅ Standby is fully caught up (received == replayed)")
    else:
        print("⚠️  Standby has unreplayed WAL — some data may be delayed")
else:
    print("\n❌ This server is NOT in standby mode!")

STANDBY STATUS CHECK
  Is in recovery (standby) mode: True
  Last WAL received:  0/65CBAE0
  Last WAL replayed:  0/65CBAE0
  Last replay time:   2026-04-20 19:20:21.487211+00:00

✅ Server is in standby mode — ready for promotion
✅ Standby is fully caught up (received == replayed)


In [6]:
# =============================================================================
# Demo: Perform Manual Failover
# =============================================================================
# WARNING: This will change your cluster topology!
# After this, the standby becomes the new primary.
#
# To restore the original setup, run:
#   docker-compose down -v && docker-compose up -d

print('=' * 65)
print('MANUAL FAILOVER PROCEDURE')
print('=' * 65)

# Step 1: Record pre-failover state
primary_conn = get_primary_connection()
primary_cur = primary_conn.cursor()
primary_cur.execute("SELECT pg_current_wal_lsn()")
pre_lsn = primary_cur.fetchone()[0]
primary_conn.close()
print(f"\nStep 1: Primary WAL position: {pre_lsn}")

# Step 2: FENCE — Stop the primary (prevent split-brain)
print("\nStep 2: FENCING — Stopping primary container...")
start_time = time.time()
subprocess.run(
    ["docker", "stop", "bcdr-pg-primary"],
    capture_output=True, timeout=30
)
print("  Primary stopped.")

# Step 3: PROMOTE — Tell the standby it is now the primary
print("\nStep 3: PROMOTING standby to primary...")
out, err = docker_exec(
    "bcdr-pg-standby",
    ["pg_ctl", "promote", "-D", "/var/lib/postgresql/data"]
)
print(f"  Result: {out or err}")

# Wait for promotion to complete
time.sleep(3)

# Step 4: VERIFY — Check the new primary accepts writes
print("\nStep 4: VERIFYING new primary...")
try:
    new_primary = psycopg2.connect(**DB_STANDBY)  # connect to old standby port
    new_primary.autocommit = True
    new_cur = new_primary.cursor()

    new_cur.execute("SELECT pg_is_in_recovery()")
    still_standby = new_cur.fetchone()[0]

    if not still_standby:
        print("  ✅ Server is no longer in recovery mode — it is the new PRIMARY!")

        # Try a write
        new_cur.execute(
            "INSERT INTO audit_log (table_name, record_id, action, changed_by) "
            "VALUES (%s, %s, %s, %s)",
            ('failover_test', 0, 'INSERT', 'failover_verification')
        )
        print("  ✅ Write succeeded on new primary!")

    failover_time = time.time() - start_time
    print(f"\n🎉 FAILOVER COMPLETE in {failover_time:.1f} seconds")
    print(f"   This is your actual RTO: {failover_time:.1f}s")
    new_primary.close()

except Exception as e:
    print(f"  ❌ Error: {e}")

print("\n📌 To restore original setup: docker-compose down -v && docker-compose up -d")

MANUAL FAILOVER PROCEDURE

Step 1: Primary WAL position: 0/65CBAE0

Step 2: FENCING — Stopping primary container...


  Primary stopped.

Step 3: PROMOTING standby to primary...
  Result: pg_ctl: cannot be run as root
Please log in (using, e.g., "su") as the (unprivileged) user that will
own the server process.



Step 4: VERIFYING new primary...

🎉 FAILOVER COMPLETE in 4.5 seconds
   This is your actual RTO: 4.5s

📌 To restore original setup: docker-compose down -v && docker-compose up -d


## 📝 Summary

### What You Learned

1. **WAL (Write-Ahead Log)** — Every database change is logged before being applied.
   Streaming replication sends these logs to the standby in real-time.

2. **Replication Lag** — The delay between primary and standby.
   Monitor it with `pg_stat_replication` and `pg_wal_lsn_diff()`.

3. **Split-Brain** — When two servers both think they are primary.
   Prevent with fencing (STONITH), quorum, or lease-based systems.

4. **Manual Failover** — Stop primary, promote standby, verify writes.
   In production, automate this with Patroni or pg_auto_failover.

### Next Notebook

In **Notebook 3**, we explore backup strategies — full, incremental,
and differential backups, plus point-in-time recovery.